# Explore R Data Structures
## Solution Notebook — Deep Dive into Vectors, Matrices, Lists & Data Frames

Complete answers, explanations of coercion hierarchy, alternate patterns, and a decision simulation.


## Flowchart

![R Data Structures Exploration Flowchart](r_data_structures_exploration_flowchart.png)


## 1. Vectors — Solutions & Insights


In [ ]:
# Exploration 1.1 — Coercion hierarchy
# Character > Numeric > Logical  (most flexible wins)

v_num_char    <- c(3.14, "GDP")
v_num_logical <- c(2.5, TRUE)
v_char_logical <- c("policy", FALSE)

print(v_num_char);    print(typeof(v_num_char))     # character
print(v_num_logical); print(typeof(v_num_logical))  # double (TRUE → 1)
print(v_char_logical);print(typeof(v_char_logical)) # character

# Comment: R coerces everything to the most flexible type present.
# Hierarchy (simplified): character > complex > numeric > integer > logical > raw


In [ ]:
# Exploration 1.2 — Named vector
inflation <- c(2.1, 2.4, 1.8, 1.2, 4.7)
names(inflation) <- 2020:2024
print(inflation["2022"])          # extract by name
print(attributes(inflation))
str(inflation)


In [ ]:
# Exploration 1.3 — Recycling
print(c(1,2,3,4) + c(10,20))   # 11 22 13 24  (no warning, length multiple)
print(c(1,2,3) + c(10,20))     # warning: longer object length is not a multiple
# R recycles the shorter vector and warns only when lengths are not multiples.


## 2. Matrices — Solutions


In [ ]:
# 2.1 byrow
m_col <- matrix(1:12, nrow = 3, ncol = 4)          # column-major (default)
m_row <- matrix(1:12, nrow = 3, ncol = 4, byrow = TRUE)
print(m_col)
print(m_row)


In [ ]:
# 2.2 Changing dimensions
m <- matrix(0, nrow = 2, ncol = 5)
print(m)
dim(m) <- c(5, 2)          # in-place reshape (data is re-laid column-wise)
print(m)
# The underlying vector is reinterpreted with the new dimensions.


## 3. Lists — Solutions


In [ ]:
# 3.1 Nested country dossier
country_dossier <- list(
  meta = list(region = "Euro Area", currency = "EUR"),
  series = list(
    inflation   = c(2.9, 2.5, 2.1),
    gdp_growth  = c(1.8, 2.0, 1.5)
  ),
  flags = c(has_forecast = TRUE, rate_decision_pending = FALSE)
)
str(country_dossier)

# Two ways to extract the inflation series
print(country_dossier$series$inflation)
print(country_dossier[["series"]][["inflation"]])


In [ ]:
# 3.2 List vs atomic vector
countries_vec  <- c("US", "EA", "JP")
countries_list <- list("US", "EA", "JP")

print(typeof(countries_vec));  print(class(countries_vec))
print(typeof(countries_list)); print(class(countries_list))
# A plain c() of characters produces an atomic character vector.
# list() always produces a list, even if all elements are the same type.


## 4. Data Frames — Solutions


In [ ]:
# 4.1 Independent column types
df <- data.frame(
  country     = c("US", "EA", "JP"),
  inflation   = c(2.5, 2.1, 0.8),
  is_advanced = c(TRUE, TRUE, TRUE),
  stringsAsFactors = FALSE
)
str(df)
print(sapply(df, typeof))
# Each column keeps its own type — this is the key advantage over matrices.


In [ ]:
# 4.2 Coercion between matrix and data.frame
m <- matrix(1:9, nrow = 3)
print(m)
dfm <- as.data.frame(m)
print(dfm)
print(as.matrix(dfm))   # column names become dimnames; all become character if mixed, numeric if pure


## 5. Structure Decision Simulation — Solution


In [ ]:
# Parameters
n_countries <- 4
n_years     <- 6
mixed_info  <- TRUE
need_math   <- FALSE

# Decision logic
if (!mixed_info && need_math) {
  recommendation <- "matrix"
  example <- matrix(rnorm(n_countries * n_years), nrow = n_countries, ncol = n_years)
} else if (mixed_info) {
  recommendation <- "data.frame (or list for deeply nested notes)"
  example <- data.frame(
    country = paste0("C", 1:n_countries),
    avg_inf = round(runif(n_countries, 1, 5), 1),
    note    = sample(c("stable", "watch", "action"), n_countries, replace = TRUE),
    stringsAsFactors = FALSE
  )
} else {
  recommendation <- "vector (single series)"
  example <- rnorm(n_years)
}

cat("Recommendation:", recommendation, "\n\n")
print(example)

# Alternate decision helper function
choose_structure <- function(mixed = FALSE, math = FALSE, n_series = 1) {
  if (n_series == 1 && !mixed) return("vector")
  if (!mixed && math) return("matrix")
  if (mixed) return("data.frame or list")
  "data.frame"
}
print(choose_structure(mixed_info, need_math, n_series = n_countries))


## Key Insights from the Exploration

1. **Coercion hierarchy** is the most common source of silent bugs. Always check `typeof()` after combining objects.
2. **Names and attributes** turn a plain vector into a self-documenting series (ideal for economic time series).
3. **Matrices** are pure and fast for linear algebra but force a single type.
4. **Lists** are the only structure that can hold truly nested, heterogeneous policy information without loss.
5. **Data frames** give you the best of both worlds for rectangular economic data: typed columns + tabular display.
6. Decision rule of thumb for economists:
   - One indicator over time → vector (named)
   - Many countries × many years, pure numeric, later algebra → matrix
   - Mixed types + notes + flags → list or data.frame
   - Final reporting table → data.frame
